# Week 4 - Day 2: Cross-Validation

**Goal:** Replace a single lucky-or-unlucky validation split with k-fold cross-validation, get a mean and standard deviation across folds, compare it to Day 1's single-split score, and confirm stratified folds are used.

**Dataset:** Breast Cancer Wisconsin dataset (same dataset as Day 1).


## Step 1: Load Data and Recreate the Day 1 Train/Test Split

Cross-validation replaces the *validation* split, not the test split — the test set is still held out separately.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

# Same split as Day 1: carve off the test set first, keep it untouched
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train+Val (used for CV):", X_train_full.shape)
print("Test (held out):        ", X_test.shape)


Train+Val (used for CV): (455, 30)
Test (held out):         (114, 30)


## Step 2: 5-Fold Cross-Validation with `cross_val_score`

Using the same `max_depth=4` chosen on Day 1, we now validate it with 5-fold cross-validation instead of a single validation split.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = RandomForestClassifier(max_depth=4, n_estimators=200, random_state=42)

scores = cross_val_score(model, X_train_full, y_train_full, cv=5, scoring="accuracy")

print("Fold scores:", scores)
print(f"Mean accuracy: {scores.mean():.4f}")
print(f"Std deviation: {scores.std():.4f}")


Fold scores: [0.95604396 0.98901099 0.92307692 0.92307692 0.94505495]
Mean accuracy: 0.9473
Std deviation: 0.0245


## Step 3: Compare to Day 1's Single-Split Validation Score

Day 1's single validation split gave **0.9561** accuracy for `max_depth=4`. Let's compare that to the cross-validated mean.

In [3]:
day1_val_score = 0.9561

print(f"Day 1 single-split validation accuracy: {day1_val_score:.4f}")
print(f"Day 2 cross-validated mean accuracy:     {scores.mean():.4f}")
print(f"Cross-validated std across folds:        {scores.std():.4f}")
print(f"Difference:                              {abs(day1_val_score - scores.mean()):.4f}")


Day 1 single-split validation accuracy: 0.9561
Day 2 cross-validated mean accuracy:     0.9473
Cross-validated std across folds:        0.0245
Difference:                              0.0088


### Reflection — Comparing the Two Estimates



The cross-validated mean and the Day 1 single-split score are close, which is reassuring — it suggests Day 1's validation split wasn't a fluke. But the cross-validated estimate is more trustworthy because it's an average over 5 different train/validation partitions instead of one. The standard deviation also gives extra information Day 1 never had: it tells us how much the accuracy would have swung if we had picked a different `random_state` for the validation split.


## Step 4: Confirm Stratified Folds Are Used

`cross_val_score` automatically uses `StratifiedKFold` for classifiers, preserving class balance in every fold. We confirm this explicitly below.

In [4]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Class balance in full training data:")
print(y_train_full.value_counts(normalize=True).round(3))

print("\nClass balance per fold (StratifiedKFold):")
for i, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full), 1):
    fold_balance = y_train_full.iloc[val_idx].value_counts(normalize=True).round(3)
    print(f"Fold {i}: {dict(fold_balance)}")


Class balance in full training data:
target
1    0.626
0    0.374
Name: proportion, dtype: float64

Class balance per fold (StratifiedKFold):
Fold 1: {1: np.float64(0.626), 0: np.float64(0.374)}
Fold 2: {1: np.float64(0.626), 0: np.float64(0.374)}
Fold 3: {1: np.float64(0.626), 0: np.float64(0.374)}
Fold 4: {1: np.float64(0.626), 0: np.float64(0.374)}
Fold 5: {1: np.float64(0.626), 0: np.float64(0.374)}


### Reflection — Why Stratification Matters Here



The Breast Cancer dataset isn't perfectly balanced (roughly 63% class 1, 37% class 0). Without stratification, plain k-fold could randomly place too many of one class in a single fold, making that fold's validation score unreliable and skewing the average. Stratified k-fold keeps the same class proportions in every fold, so each fold is a fair, representative test of the model — this directly addresses the imbalance concern raised in Week 3.


## Summary

| Approach | Estimate | Extra Info |
|---|---|---|
| Day 1: single validation split | 0.9561 | None — one number, no sense of stability |
| Day 2: 5-fold cross-validation | mean ± std (see Step 2 output) | Standard deviation shows how stable the estimate is |

- Cross-validation uses every training point for validation exactly once, so no single split can dominate the result.
- Stratified k-fold (used automatically here) keeps class balance consistent across folds — important since this dataset is not perfectly balanced.
- The test set from Day 1 was never touched in this notebook — it's reserved for the final evaluation once tuning is fully complete.
